# FoodLensVN — Evaluation (A1 / A2 / B1 / B2)

Run after `train_a1_a2.ipynb` and `train_b2.ipynb` have produced and pushed their checkpoints/adapter to HF (`Tamir39/foodlensvn-A1`, `-A2`, `-B2`).

What this notebook does:
1. Clone repo + install deps + HF login.
2. Fetch dataset + build processed splits (eval reads `data/processed/annotations/test.json`).
3. Pull A1/A2 checkpoints + B2 adapter from HF.
4. Run `scripts/eval.py` for all four configs — writes `reports/<name>_metrics.json` and `reports/<name>_errors.json`.
5. Print a side-by-side summary table for the report.

Eval artifacts stay local to the Kaggle session — copy them into the report manually. No HF push.

**Setup before running:**
1. **Settings → Accelerator → GPU** (P100 or T4 ×2).
2. **Settings → Internet → On**.
3. **Add-ons → Secrets → `HF_TOKEN`** (read access is enough).

In [ ]:
# Cell 1: Clone the develop branch (or pull latest)
import os
%cd /kaggle/working/
REPO_URL = 'https://github.com/tamir39/vqa-viet-project.git'
REPO_DIR = 'vqa-viet-project'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch develop {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}

In [ ]:
# Cell 2: Install deps via uv
!pip install -q uv
!uv sync --frozen 2>&1 | tail -10

In [ ]:
# Cell 3: HF login via Kaggle secret
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print('HF login OK')

In [ ]:
# Cell 4: Path + env setup
import os, sys
ROOT = '/kaggle/working/vqa-viet-project'
os.environ['FOODLENS_DATA_DIR'] = f'{ROOT}/data/foodlensvn'
os.environ['MPLBACKEND'] = 'Agg'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
# Cell 5: Pull dataset + build processed splits (test.json is what eval reads)
!uv run python scripts/fetch_dataset.py --dest $FOODLENS_DATA_DIR
!uv run python scripts/build_dataset.py --data-dir $FOODLENS_DATA_DIR --output-dir data/processed --image-variant squared

In [ ]:
# Cell 6: Pull A1/A2 checkpoints + B2 adapter from HF
from huggingface_hub import snapshot_download
for name in ['A1', 'A2', 'B2']:
    snapshot_download(
        repo_id=f'Tamir39/foodlensvn-{name}',
        repo_type='model',
        local_dir=f'reports/{name}',
        token=os.environ['HF_TOKEN'],
    )
    print(f'{name}: pulled to reports/{name}/')

In [ ]:
# Cell 7: Eval A1 + A2 (modular — need --checkpoint)
!uv run python scripts/eval.py --config configs/A1.yaml --checkpoint reports/A1/checkpoints/best.pt
!uv run python scripts/eval.py --config configs/A2.yaml --checkpoint reports/A2/checkpoints/best.pt

In [ ]:
# Cell 8: Eval B1 (zero-shot — no checkpoint)
!uv run python scripts/eval.py --config configs/B1.yaml

In [ ]:
# Cell 9: Eval B2 (LoRA — needs --adapter)
!uv run python scripts/eval.py --config configs/B2.yaml --adapter reports/B2/adapter

In [ ]:
# Cell 10: Side-by-side overall metrics for the report
import json
from pathlib import Path

rows = []
for name in ['A1', 'A2', 'B1', 'B2']:
    p = Path(f'reports/{name}_metrics.json')
    if not p.exists():
        print(f'skip {name}: {p} not found')
        continue
    o = json.loads(p.read_text(encoding='utf-8'))['overall']
    rows.append((name, o.get('exact', 0), o.get('soft', 0), o.get('bleu', 0),
                 o.get('rouge_l', 0), o.get('meteor', 0), o.get('bertscore_f1', 0)))

print(f'{"config":<6} {"exact":>7} {"soft":>7} {"bleu":>7} {"rougeL":>7} {"meteor":>7} {"bertF1":>7}')
for r in rows:
    print(f'{r[0]:<6} {r[1]:>7.3f} {r[2]:>7.3f} {r[3]:>7.3f} {r[4]:>7.3f} {r[5]:>7.3f} {r[6]:>7.3f}')